# nb01b - Enrichment: two join-ready reference tables

Two joins enrich the market-share analysis, each at its own grain (kept separate on purpose):

1. **County population** -> Medi-Cal penetration rate (how much of each county relies on Medi-Cal). County grain.
2. **Plan quality (HEDIS AQFS)** -> market share versus quality. Plan grain.

This notebook writes the two clean reference files with keys aligned to the enrollment data, so the joins performed in Tableau are clean. The joins themselves are done in Tableau (that is the Connect and Transform exam skill).

In [1]:
from pathlib import Path
import pandas as pd
DATA = Path('..') / 'data'

## 1. County population reference (for the penetration join)

California Department of Finance E-2 county population estimates, July 2024 (rounded to the nearest thousand). Reference input, not a computed figure, so it is keyed in here and cited, the same way the CMS measure weights were in the Star Ratings project.

**To refresh:** replace with the current DOF E-2 file from https://dof.ca.gov/forecasting/demographics/estimates/E-2/ . The join key is the county name, which already matches the enrollment file.

In [2]:
# CA DOF E-2, July 2024 county population estimates (people)
CA_COUNTY_POP_2024 = {
    'Los Angeles': 9663000, 'San Diego': 3270000, 'Orange': 3142000, 'Riverside': 2529000,
    'San Bernardino': 2214000, 'Santa Clara': 1918000, 'Alameda': 1649000, 'Sacramento': 1611000,
    'Contra Costa': 1166000, 'Fresno': 1024000, 'Kern': 922000, 'Ventura': 835000,
    'San Francisco': 827000, 'San Joaquin': 810000, 'San Mateo': 730000, 'Stanislaus': 556000,
    'Tulare': 483000, 'Sonoma': 482000, 'Solano': 455000, 'Santa Barbara': 444000,
    'Monterey': 436000, 'Placer': 428000, 'Merced': 296000, 'San Luis Obispo': 282000,
    'Santa Cruz': 262000, 'Marin': 256000, 'Yolo': 224000, 'Butte': 208000,
    'El Dorado': 192000, 'Shasta': 182000, 'Imperial': 180000, 'Madera': 165000,
    'Kings': 154000, 'Napa': 134000, 'Humboldt': 133000, 'Nevada': 102000,
    'Sutter': 100000, 'Mendocino': 89000, 'Yuba': 87000, 'San Benito': 69000,
    'Lake': 68000, 'Tehama': 65000, 'Tuolumne': 54000, 'Calaveras': 46000,
    'Siskiyou': 43000, 'Amador': 42000, 'Glenn': 29000, 'Lassen': 28000,
    'Del Norte': 27000, 'Colusa': 22000, 'Inyo': 19000, 'Plumas': 19000,
    'Mariposa': 17000, 'Trinity': 16000, 'Mono': 13000, 'Modoc': 8700,
    'Sierra': 3200, 'Alpine': 1200,
}
pop = pd.DataFrame(sorted(CA_COUNTY_POP_2024.items()), columns=['County', 'Population'])
pop['Pop Year'] = 2024

# every county in the enrollment data must have a population, or the map has holes
enr = pd.read_csv(DATA / 'ca_county_enrollment_clean.csv')
missing = set(enr.County) - set(pop.County)
assert not missing, f'counties with no population: {missing}'
pop.to_csv(DATA / 'ca_county_population.csv', index=False)
print('wrote ca_county_population.csv', pop.shape, '| all enrollment counties matched')

wrote ca_county_population.csv (58, 3) | all enrollment counties matched


### Sanity check: penetration = Medi-Cal enrollees / population (validation only)

This is a **validation** step, not the analysis join. It confirms every county population is plausible before Tableau uses it. It writes no joined file. The enrollment-to-population join is done in Tableau, on the county key.

Penetration should be roughly a quarter to a half in most counties (statewide Medi-Cal covers about a third of Californians), higher in lower income counties. If any county is implausible (over 100%, or near zero), the population figure is wrong.

In [3]:
# VALIDATION ONLY. This merge checks that every population figure is plausible.
# It writes nothing. The real enrollment-to-population join is performed in TABLEAU,
# on the County key, so the reference files above stay separate and join-ready.
chk = enr.merge(pop, on='County')
chk['Penetration'] = (chk['Medi-Cal MC Enrollees'] / chk['Population']).round(3)
chk = chk.sort_values('Medi-Cal MC Enrollees', ascending=False)
print('penetration range:', chk.Penetration.min(), 'to', chk.Penetration.max())
print()
print('=== top 10 counties by enrollment, with penetration ===')
show = chk.head(10)[['County', 'Medi-Cal MC Enrollees', 'Population', 'Penetration']].copy()
show['Medi-Cal MC Enrollees'] = show['Medi-Cal MC Enrollees'].map(lambda v: f'{int(v):,}')
show['Population'] = show['Population'].map(lambda v: f'{int(v):,}')
show['Penetration'] = (show['Penetration']*100).round(1).astype(str) + '%'
print(show.to_string(index=False))
assert chk.Penetration.between(0.05, 0.75).all(), 'a penetration rate looks wrong; check that county population'

penetration range: 0.183 to 0.551

=== top 10 counties by enrollment, with penetration ===
        County Medi-Cal MC Enrollees Population Penetration
   Los Angeles             3,521,674  9,663,000       36.4%
San Bernardino               866,361  2,214,000       39.1%
     Riverside               862,488  2,529,000       34.1%
        Orange               855,944  3,142,000       27.2%
     San Diego               821,390  3,270,000       25.1%
    Sacramento               560,275  1,611,000       34.8%
        Fresno               473,382  1,024,000       46.2%
          Kern               448,377    922,000       48.6%
       Alameda               442,000  1,649,000       26.8%
   Santa Clara               397,829  1,918,000       20.7%


## 2. Plan quality reference (for the share versus quality join)

The composite Medi-Cal quality score (AQFS) from the HEDIS project (Post 2), for the Los Angeles region. Plan codes are mapped to the same brand names used in the market-share file so the Tableau join matches on `Brand`.

In [4]:
aqfs = pd.read_csv(Path('..') / '..' / 'tableau_hedis' / 'data' / 'aqfs_clean.csv')
la_q = aqfs[aqfs.Region == 'Los Angeles'].copy()

# align the HEDIS plan codes to the market-share brand names
BRAND = {'LA Care': 'L.A. Care', 'Health Net': 'Health Net', 'Kaiser': 'Kaiser Permanente'}
la_q['Brand'] = la_q.Plan.map(BRAND)
la_q = la_q.dropna(subset=['Brand'])[['Year', 'Brand', 'AQFS']].sort_values(['Year', 'Brand'])
la_q.to_csv(DATA / 'la_plan_quality.csv', index=False)
print('wrote la_plan_quality.csv', la_q.shape)
print()
print('=== LA plan quality by year (AQFS, higher is better) ===')
print(la_q.pivot(index='Year', columns='Brand', values='AQFS').to_string())
print()
print('Latest year, for the share vs quality scatter:')
print(la_q[la_q.Year == la_q.Year.max()].to_string(index=False))

wrote la_plan_quality.csv (16, 3)

=== LA plan quality by year (AQFS, higher is better) ===
Brand  Health Net  L.A. Care
Year                        
2016        61.36      60.91
2017        63.33      66.67
2018        65.24      66.67
2019        62.63      66.84
2020        60.00      72.22
2021        52.63      59.47
2022        57.33      67.33
2023        52.00      59.33

Latest year, for the share vs quality scatter:
 Year      Brand  AQFS
 2023 Health Net 52.00
 2023  L.A. Care 59.33
